<center><h1>Liu_Peiyi_HW3</h1></center>
<br>
<br>

Name: Peiyi Liu
<br>
Github Username: Eternal-799
<br>
USC ID: 2086699860

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [8]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Get the AReM Data Set

In [9]:
# Set the directory containing the unzipped AReM dataset folders
data_dir = '../data/AReM'

def read_arem_csv(file_path):
    """
    Robust reader for AReM dataset files that handles irregular headers,
    differing skip-row lengths, and mixed comma/whitespace delimiters.
    """
    # Try reading as comma-separated first
    try:
        # skiprows=4 handles normal metadata headers
        df = pd.read_csv(file_path, skiprows=4, header=None, on_bad_lines='skip')
        # If fewer than 6 columns read, it is likely whitespace-delimited
        if df.shape[1] < 6:
            df = pd.read_csv(file_path, skiprows=4, sep=r'\s+', header=None, on_bad_lines='skip')
    except Exception:
        df = pd.read_csv(file_path, skiprows=4, sep=r'\s+', header=None, on_bad_lines='skip')

    # Convert all content to numeric, coercing text/headers into NaN
    df = df.apply(pd.to_numeric, errors='coerce').dropna(how='all')

    # If the file had 7 or 8 columns (e.g. index/time columns), keep the last 6 columns
    if df.shape[1] >= 6:
        df = df.iloc[:, -6:]
    else:
        # Fallback: re-read the entire raw file line-by-line ignoring non-data lines
        valid_rows = []
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                # Replace commas with whitespace and split
                parts = line.replace(',', ' ').strip().split()
                if len(parts) >= 6:
                    try:
                        nums = [float(x) for x in parts[-6:]]
                        valid_rows.append(nums)
                    except ValueError:
                        continue
        df = pd.DataFrame(valid_rows)

    df.columns = ['avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'avg_rss23', 'var_rss23']
    return df

### (b) Test and Train Data

In [10]:
# List of all 7 activity classes
activities = ['bending1', 'bending2', 'cycling', 'lying', 'sitting', 'standing', 'walking']

train_files = []
test_files = []

for act in activities:
    folder_path = os.path.join(data_dir, act)
    # Collect all dataset*.csv files in the activity directory
    csv_files = sorted(glob.glob(os.path.join(folder_path, 'dataset*.csv')))
    
    # Sort files naturally by index (dataset1, dataset2, ...)
    csv_files = sorted(csv_files, key=lambda x: int(os.path.basename(x).replace('dataset', '').replace('.csv', '')))
    
    # Keep instances 1 and 2 for bending folders; instances 1, 2, and 3 for others as test data
    test_count = 2 if 'bending' in act else 3
    
    test_files.extend(csv_files[:test_count])
    train_files.extend(csv_files[test_count:])

# Verify dataset splits
print(f"Number of training files: {len(train_files)} (Expected: 69)")
print(f"Number of testing files: {len(test_files)} (Expected: 19)")

Number of training files: 69 (Expected: 69)
Number of testing files: 19 (Expected: 19)


### (c) Feature Extraction

#### i. Research

Common time-domain features used in time series classification include:
- **Measures of Central Tendency**: Mean, Median, Mode.
- **Measures of Dispersion / Spread**: Minimum, Maximum, Range, Standard Deviation, Variance, Interquartile Range (IQR), 1st Quartile (25th percentile), 3rd Quartile (75th percentile).
- **Shape & Distribution**: Skewness, Kurtosis.
- **Energy & Magnitude**: Root Mean Square (RMS), Signal Energy, Peak-to-Peak Amplitude.

#### ii. Extraction

In [11]:
# Function to extract the 7 time-domain features for each of the 6 time series in a file
def extract_features(file_path):
    df = read_arem_csv(file_path)
    features = {}
    
    # Process each of the 6 columns
    for col_idx, col in enumerate(df.columns, start=1):
        series = df[col]
        features[f'min_{col_idx}'] = series.min()
        features[f'max_{col_idx}'] = series.max()
        features[f'mean_{col_idx}'] = series.mean()
        features[f'median_{col_idx}'] = series.median()
        features[f'std_{col_idx}'] = series.std(ddof=1)
        features[f'1st_quart_{col_idx}'] = series.quantile(0.25)
        features[f'3rd_quart_{col_idx}'] = series.quantile(0.75)
        
    return features

# Gather all 88 files across all activities
all_files = []
for act in activities:
    folder_path = os.path.join(data_dir, act)
    files = sorted(glob.glob(os.path.join(folder_path, 'dataset*.csv')),
                   key=lambda x: int(os.path.basename(x).replace('dataset', '').replace('.csv', '')))
    all_files.extend(files)

# Extract features for all instances
features_list = [extract_features(fp) for fp in all_files]

# Create DataFrame
feature_df = pd.DataFrame(features_list)
feature_df.index = np.arange(1, len(feature_df) + 1)
feature_df.index.name = 'Instance'

print(f"Dataset shape: {feature_df.shape} (Expected: (88, 42))")
feature_df

Dataset shape: (88, 42) (Expected: (88, 42))


,min_1,max_1,mean_1,median_1,std_1,1st_quart_1,3rd_quart_1,min_2,max_2,mean_2,...,std_5,1st_quart_5,3rd_quart_5,min_6,max_6,mean_6,median_6,std_6,1st_quart_6,3rd_quart_6
Instance,,,,,,,,,,,,,,,,,,,,,
1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,0.0,1.30,0.358604,...,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,0.0,1.22,0.372437,...,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,0.0,1.70,0.426250,...,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,0.0,3.00,0.696042,...,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,0.0,2.83,0.535979,...,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,0.0,14.67,4.576562,...,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
85,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,0.0,13.47,4.456333,...,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
86,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,0.0,12.47,4.371958,...,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350


#### iii. Standard Deviation

In [12]:
# 1. Ensure all columns are purely numeric float types
feature_df = feature_df.apply(pd.to_numeric, errors='coerce')
std_estimates = feature_df.std(axis=0, ddof=1)

# 2. Bootstrap function to compute the 90% confidence interval for standard deviation
def get_bootstrap_std_ci(data, n_iterations=1000, ci=0.90, random_state=42):
    rng = np.random.default_rng(random_state)
    
    # Strip any potential NaN values to ensure numeric validity
    clean_data = np.asarray(data, dtype=float)
    clean_data = clean_data[~np.isnan(clean_data)]
    n = len(clean_data)
    
    # For constant features with zero variance, return interval (0.0, 0.0)
    if n == 0 or np.all(clean_data == clean_data[0]):
        return 0.0, 0.0
    
    # Sample with replacement and compute standard deviation
    boot_stds = []
    for _ in range(n_iterations):
        sample = rng.choice(clean_data, size=n, replace=True)
        boot_stds.append(np.std(sample, ddof=1))
        
    boot_stds = np.array(boot_stds)
    
    # 90% confidence interval corresponds to 5th and 95th percentiles
    lower = float(np.percentile(boot_stds, 5.0))
    upper = float(np.percentile(boot_stds, 95.0))
    return lower, upper

# 3. Compute bootstrap confidence intervals for all 42 features
ci_data = []
for col in feature_df.columns:
    col_vals = feature_df[col].to_numpy(dtype=float)
    est_std = float(std_estimates[col])
    lower, upper = get_bootstrap_std_ci(col_vals, n_iterations=1000, ci=0.90)
    ci_data.append({
        'Feature': col,
        'Estimated Std': est_std,
        '90% CI Lower': lower,
        '90% CI Upper': upper
    })

ci_df = pd.DataFrame(ci_data)
ci_df

,Feature,Estimated Std,90% CI Lower,90% CI Upper
0,min_1,9.624011,8.312613,10.879231
1,max_1,4.207745,3.196574,5.142660
2,mean_1,5.276413,4.666547,5.868645
3,median_1,5.386624,4.752269,5.964439
4,std_1,1.771313,1.576753,1.946868
5,1st_quart_1,6.128143,5.535663,6.619807
6,3rd_quart_1,5.031028,4.275429,5.789852
7,min_2,0.000000,0.000000,0.000000
8,max_2,5.059656,4.604158,5.381310
9,mean_2,1.577941,1.397025,1.707114


#### iv. Select Features

We select **Minimum (`min`)**, **Mean (`mean`)**, and **Maximum (`max`)** as the three most important time-domain features:
1. **Minimum (`min`)**: Represents the baseline sensor level and lower boundary of the motion.
2. **Mean (`mean`)**: Reflects the overall central tendency and steady-state intensity of the activity.
3. **Maximum (`max`)**: Captures peak movements, acceleration bursts, and upper bounds of motion intensity.

Together, these three features effectively capture both the location (central tendency) and span (range) of the signals while being robust and computationally efficient.

## 2. ISLR 3.7.4

### (a) Linear Train

**Answer**: The training RSS for the cubic regression will be lower than (or at most equal to) the training RSS for the linear regression.

**Reasoning**: Cubic regression is more flexible than linear regression and contains the linear model as a special case ($\beta_2 = \beta_3 = 0$). Ordinary Least Squares (OLS) minimizes the sum of squared residuals on the training data. Therefore, with higher degrees of freedom, the cubic model can fit the training points better or at least as well as the linear model, resulting in a lower (or equal) training RSS.

### (b) Linear Test

**Answer**: The test RSS for the linear regression will be lower than the test RSS for the cubic regression.

**Reasoning**: Since the true underlying relationship is linear, the linear regression model has virtually zero bias while maintaining a low variance. In contrast, the cubic model is overly flexible and overfits the irreducible noise in the training set, leading to higher estimation variance. Consequently, the cubic model will have a higher test RSS.

### (c) Not Linear Train

**Answer**: The training RSS for the cubic regression will still be lower than (or at most equal to) that of the linear regression.

**Reasoning**: Regardless of what the true relationship is, cubic regression has more flexibility and more parameters to fit the training observations. It will always achieve a training RSS that is less than or equal to that of the linear model.

### (d) Not Linear Testing

**Answer**: There is not enough information to definitively tell; it depends on how non-linear the true relationship is.

**Reasoning**:
- If the true underlying relationship is significantly non-linear (e.g., cubic or higher order), the reduction in bias from using the cubic model will outweigh the increase in variance, leading to a lower test RSS for cubic regression.
- However, if the true relationship is only slightly non-linear, the slight decrease in bias may not compensate for the larger variance introduced by the cubic model, in which case linear regression could still have a lower test RSS.

## 3. ISLR 3.7.3 - Extra Practice 

**Model Equation**:
$$\hat{Y} = 50 + 20 \times \text{GPA} + 0.07 \times \text{IQ} + 35 \times \text{Level} + 0.01 \times (\text{GPA} \times \text{IQ}) - 10 \times (\text{GPA} \times \text{Level})$$
where $\text{Level} = 1$ for College and $\text{Level} = 0$ for High School.

- For High School graduates ($\text{Level} = 0$):
  $$\hat{Y}_{\text{HS}} = 50 + 20 \times \text{GPA} + 0.07 \times \text{IQ} + 0.01 \times (\text{GPA} \times \text{IQ})$$

- For College graduates ($\text{Level} = 1$):
  $$\hat{Y}_{\text{College}} = 85 + 10 \times \text{GPA} + 0.07 \times \text{IQ} + 0.01 \times (\text{GPA} \times \text{IQ})$$

- Difference ($\hat{Y}_{\text{College}} - \hat{Y}_{\text{HS}}$):
  $$\hat{Y}_{\text{College}} - \hat{Y}_{\text{HS}} = 35 - 10 \times \text{GPA}$$

---

#### (a) Which statement is correct?
**Answer**: **iii is correct.** For a fixed value of IQ and GPA, high school graduates earn more, on average, than college graduates provided that the GPA is high enough.

**Reasoning**:
Setting $\hat{Y}_{\text{College}} < \hat{Y}_{\text{HS}} \iff 35 - 10 \times \text{GPA} < 0 \iff \text{GPA} > 3.5$.
Thus, high school graduates earn more than college graduates when GPA exceeds 3.5.

---

#### (b) Predict the salary of a college graduate with IQ = 110 and GPA = 4.0
**Answer**:
$$\hat{Y} = 85 + 10(4.0) + 0.07(110) + 0.01(4.0 \times 110) = 85 + 40 + 7.7 + 4.4 = 137.1$$
The predicted starting salary is **$137,100** (or 137.1 in thousands of dollars).

---

#### (c) Is the interaction effect small because its coefficient is 0.01?
**Answer**: **False.**
The magnitude of the coefficient depends entirely on the units of the predictors. Since IQ takes values around 100 and GPA around 3-4, their product $\text{GPA} \times \text{IQ}$ can be around 300 to 400, meaning a coefficient of 0.01 still contributes non-trivially to the prediction. To determine whether the interaction term is truly significant, we must examine its p-value or t-statistic, not just the raw magnitude of the coefficient.

## 3. ISLR 3.7.5 - Extra Practice 

Consider linear regression without an intercept:
$$\hat{y}_i = x_i \hat{\beta}, \quad \text{where} \quad \hat{\beta} = \frac{\sum_{j=1}^n x_j y_j}{\sum_{k=1}^n x_k^2}$$

Substituting $\hat{\beta}$ into $\hat{y}_i$:
$$\hat{y}_i = x_i \left( \frac{\sum_{j=1}^n x_j y_j}{\sum_{k=1}^n x_k^2} \right) = \sum_{j=1}^n \left( \frac{x_i x_j}{\sum_{k=1}^n x_k^2} \right) y_j$$

Therefore, we can write:
$$\hat{y}_i = \sum_{j=1}^n a_j y_j, \quad \text{where} \quad a_j = \frac{x_i x_j}{\sum_{k=1}^n x_k^2}$$